In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('merge/result/C2_XGB_예측.csv')

df

,ID,Predicted_Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,D
3,TEST_00003,E
4,TEST_00004,E
...,...,...
599995,TEST_99995,E
599996,TEST_99996,E
599997,TEST_99997,E
599998,TEST_99998,C


In [3]:
df = df.rename(columns={'Predicted_Segment': 'Segment'})

In [4]:
# 각 ID별로 Segment 최빈값 구하기
mode_segment_per_id = (
    df.groupby('ID')['Segment']
    .agg(lambda x: x.mode().iloc[0]) 
    .reset_index()
    .rename(columns={'Segment': 'Segment_mode'})
)

# 원본 데이터와 결합 (각 row에 ID별 최빈 Segment 정보 붙이기)
df_merged = df.merge(mode_segment_per_id, on='ID')

# ID별로 최빈 Segment와 같은 Segment만 필터링
df_filtered = df_merged[df_merged['Segment'] == df_merged['Segment_mode']]

# 그중에서 ID당 1개 row만 남기기
df_final = df_filtered.drop_duplicates(subset='ID', keep='first')

# 불필요한 보조 컬럼 제거 (Segment_mode)
df_final = df_final.drop(columns=['Segment_mode']).reset_index(drop=True)

In [5]:
df_final

,ID,Segment
0,TEST_00000,E
1,TEST_00001,E
2,TEST_00002,D
3,TEST_00003,E
4,TEST_00004,E
...,...,...
99995,TEST_98246,D
99996,TEST_98533,E
99997,TEST_98566,E
99998,TEST_99853,C


In [6]:
df_final.to_csv('merge/result/C3_XGB_예측_중복제거(최빈값).csv', index=False, encoding='utf-8-sig')